# Improved eLCS — 5 Runs

Runs the improved eLCS **5 times** with different random states.

- 5,000 training rows per run
- 2:1 No-Rain : Rain training ratio
- `N = 1000`
- `learning_iterations = 5000`
- same untouched test set for all runs
- reports mean and standard deviation


In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from skeLCS.eLCS import eLCS

data_path = Path("../data/weatherAUS.csv")
results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)

run_seeds = [42, 52, 62, 72, 82]
sample_size = 5000
no_rain_size = 3333
rain_size = 1667
population_size = 1000
learning_iterations = 5000

print("Seeds:", run_seeds)
print("Sample size:", sample_size)
print("N:", population_size)


## 1. Fixed 80/20 split

In [ ]:
df = pd.read_csv(data_path)

df = df.dropna(subset=["RainTomorrow"]).copy()

df["RainTomorrow"] = (
    df["RainTomorrow"]
    .astype(str)
    .str.strip()
    .map({"No": 0, "Yes": 1})
)

if df["RainTomorrow"].isna().any():
    raise ValueError("Unexpected RainTomorrow value found.")

df["RainTomorrow"] = df["RainTomorrow"].astype(int)

x = df.drop(columns=["RainTomorrow"])
y = df["RainTomorrow"]

x_train_raw, x_test_raw, y_train_raw, y_test_raw = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train:", len(x_train_raw))
print("Test:", len(x_test_raw))
print(y_train_raw.value_counts().sort_index())
print(y_test_raw.value_counts().sort_index())


## 2. Leakage-free preprocessing

In [ ]:
clean_x_train = x_train_raw.copy()
clean_x_test = x_test_raw.copy()

for col in clean_x_train.select_dtypes(include="object").columns:
    clean_x_train[col] = clean_x_train[col].astype(str).str.strip()
    clean_x_test[col] = clean_x_test[col].astype(str).str.strip()

numeric_columns = [
    "MinTemp", "MaxTemp", "Rainfall", "Evaporation", "Sunshine",
    "WindGustSpeed", "WindSpeed9am", "WindSpeed3pm",
    "Humidity9am", "Humidity3pm",
    "Pressure9am", "Pressure3pm",
    "Cloud9am", "Cloud3pm",
    "Temp9am", "Temp3pm"
]

categorical_columns = [
    "WindGustDir",
    "WindDir9am",
    "WindDir3pm",
    "RainToday"
]

train_medians = clean_x_train[numeric_columns].median()

clean_x_train[numeric_columns] = clean_x_train[numeric_columns].fillna(train_medians)
clean_x_test[numeric_columns] = clean_x_test[numeric_columns].fillna(train_medians)

for col in categorical_columns:
    clean_x_train[col] = clean_x_train[col].replace("nan", np.nan)
    clean_x_test[col] = clean_x_test[col].replace("nan", np.nan)

    mode_value = clean_x_train[col].mode(dropna=True)[0]

    clean_x_train[col] = clean_x_train[col].fillna(mode_value)
    clean_x_test[col] = clean_x_test[col].fillna(mode_value)

for col in ["Cloud9am", "Cloud3pm"]:
    clean_x_train.loc[clean_x_train[col] > 8, col] = 8
    clean_x_test.loc[clean_x_test[col] > 8, col] = 8

evaporation_median = train_medians["Evaporation"]

train_anomaly = (
    (clean_x_train["Location"] == "Williamtown")
    & (clean_x_train["Evaporation"] == 145)
)

test_anomaly = (
    (clean_x_test["Location"] == "Williamtown")
    & (clean_x_test["Evaporation"] == 145)
)

clean_x_train.loc[train_anomaly, "Evaporation"] = evaporation_median
clean_x_test.loc[test_anomaly, "Evaporation"] = evaporation_median

for current in [clean_x_train, clean_x_test]:
    current["Date"] = pd.to_datetime(current["Date"])
    current["Year"] = current["Date"].dt.year
    current["Month"] = current["Date"].dt.month
    current["Day"] = current["Date"].dt.day
    current.drop(columns=["Date"], inplace=True)

rain_today_map = {"No": 0, "Yes": 1}

clean_x_train["RainToday"] = clean_x_train["RainToday"].map(rain_today_map)
clean_x_test["RainToday"] = clean_x_test["RainToday"].map(rain_today_map)

direction_degrees = {
    "N": 0.0, "NNE": 22.5, "NE": 45.0, "ENE": 67.5,
    "E": 90.0, "ESE": 112.5, "SE": 135.0, "SSE": 157.5,
    "S": 180.0, "SSW": 202.5, "SW": 225.0, "WSW": 247.5,
    "W": 270.0, "WNW": 292.5, "NW": 315.0, "NNW": 337.5
}

wind_columns = ["WindGustDir", "WindDir9am", "WindDir3pm"]

for col in wind_columns:
    train_angle = np.deg2rad(clean_x_train[col].map(direction_degrees))
    test_angle = np.deg2rad(clean_x_test[col].map(direction_degrees))

    clean_x_train[f"{col}_sin"] = np.sin(train_angle)
    clean_x_train[f"{col}_cos"] = np.cos(train_angle)

    clean_x_test[f"{col}_sin"] = np.sin(test_angle)
    clean_x_test[f"{col}_cos"] = np.cos(test_angle)

clean_x_train.drop(columns=wind_columns, inplace=True)
clean_x_test.drop(columns=wind_columns, inplace=True)

location_values = sorted(clean_x_train["Location"].dropna().unique())

location_map = {
    location: index
    for index, location in enumerate(location_values)
}

clean_x_train["Location"] = clean_x_train["Location"].map(location_map)
clean_x_test["Location"] = clean_x_test["Location"].map(location_map)

if clean_x_test["Location"].isna().any():
    raise ValueError("Unseen test Location found.")

test_row_ids = clean_x_test.index.to_numpy()

x_train_clean = clean_x_train.reset_index(drop=True).copy()
x_test_clean = clean_x_test.reset_index(drop=True).copy()

y_train_clean = y_train_raw.reset_index(drop=True).copy()
y_test_clean = y_test_raw.reset_index(drop=True).copy()

print("Train shape:", x_train_clean.shape)
print("Test shape:", x_test_clean.shape)
print("Train missing:", int(x_train_clean.isna().sum().sum()))
print("Test missing:", int(x_test_clean.isna().sum().sum()))
print("Non-numeric:", x_train_clean.select_dtypes(exclude=np.number).columns.tolist())


## 3. Run improved eLCS five times

In [ ]:
train_data = x_train_clean.copy()
train_data["RainTomorrow"] = y_train_clean.to_numpy()

discrete_columns = ["Location", "RainToday"]

discrete_attributes = np.array([
    x_train_clean.columns.get_loc(col)
    for col in discrete_columns
])

run_results = []

for run_number, seed in enumerate(run_seeds, start=1):

    print("=" * 70)
    print(f"RUN {run_number} / 5 | random_state={seed}")

    no_rain_sample = (
        train_data[train_data["RainTomorrow"] == 0]
        .sample(n=no_rain_size, random_state=seed, replace=False)
    )

    rain_sample = (
        train_data[train_data["RainTomorrow"] == 1]
        .sample(n=rain_size, random_state=seed, replace=False)
    )

    run_sample = (
        pd.concat([no_rain_sample, rain_sample])
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )

    x_run = run_sample.drop(columns=["RainTomorrow"])
    y_run = run_sample["RainTomorrow"]

    model = eLCS(
        learning_iterations=learning_iterations,
        N=population_size,
        discrete_attribute_limit="d",
        specified_attributes=discrete_attributes,
        random_state=seed
    )

    start_time = time.perf_counter()

    _ = model.fit(
        x_run.to_numpy(dtype=float),
        y_run.to_numpy(dtype=float)
    )

    predictions = model.predict(
        x_test_clean.to_numpy(dtype=float)
    )

    runtime_minutes = (time.perf_counter() - start_time) / 60

    accuracy = accuracy_score(y_test_clean, predictions)
    balanced_accuracy = balanced_accuracy_score(y_test_clean, predictions)

    rain_precision = precision_score(
        y_test_clean,
        predictions,
        pos_label=1,
        zero_division=0
    )

    rain_recall = recall_score(
        y_test_clean,
        predictions,
        pos_label=1,
        zero_division=0
    )

    rain_f1 = f1_score(
        y_test_clean,
        predictions,
        pos_label=1,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_test_clean,
        predictions,
        labels=[0, 1]
    ).ravel()

    run_results.append({
        "Run": run_number,
        "Random State": seed,
        "Accuracy": accuracy,
        "Balanced Accuracy": balanced_accuracy,
        "Rain Precision": rain_precision,
        "Rain Recall": rain_recall,
        "Rain F1": rain_f1,
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "Runtime Minutes": runtime_minutes
    })

    pd.DataFrame({
        "row_id": test_row_ids,
        "actual": y_test_clean.to_numpy(),
        "prediction": predictions
    }).to_csv(
        results_dir / f"improved_elcs_5run_predictions_seed_{seed}.csv",
        index=False
    )

    print(
        f"Accuracy={accuracy:.4f} | "
        f"BalancedAcc={balanced_accuracy:.4f} | "
        f"RainPrecision={rain_precision:.4f} | "
        f"RainRecall={rain_recall:.4f} | "
        f"RainF1={rain_f1:.4f}"
    )

results = pd.DataFrame(run_results)

results.to_csv(
    results_dir / "improved_elcs_5run_results.csv",
    index=False
)

print("\nAll five runs complete.")
print(results.round(4).to_string(index=False))


## 4. Five-run average

In [ ]:
metric_columns = [
    "Accuracy",
    "Balanced Accuracy",
    "Rain Precision",
    "Rain Recall",
    "Rain F1"
]

summary = pd.DataFrame({
    "Metric": metric_columns,
    "Mean": [results[col].mean() for col in metric_columns],
    "Standard Deviation": [results[col].std(ddof=1) for col in metric_columns],
    "Minimum": [results[col].min() for col in metric_columns],
    "Maximum": [results[col].max() for col in metric_columns]
})

summary.to_csv(
    results_dir / "improved_elcs_5run_summary.csv",
    index=False
)

print(summary.round(4).to_string(index=False))

print("\nREPORT VALUES")
print("-" * 40)

for metric in metric_columns:
    mean_value = results[metric].mean()
    std_value = results[metric].std(ddof=1)

    print(
        f"{metric}: {mean_value:.4f} "
        f"(SD={std_value:.4f})"
    )
